In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
def f(x):
    return 3*x**2 - 4*x + 5

In [ ]:
f(3.0)

In [ ]:
xs = np.arange(-5, 5, 0.25)
ys = f(xs)
plt.plot(xs, ys)

In [ ]:
h = 0.00001
x = 2/3
(f(x + h) - f(x))/h

In [ ]:
# lets get more complex
a = 2.0
b = -3.0
c = 10
d = a*b + c
print(d)

In [ ]:
h = 0.00001

# inputs
a = 2.0
b = -3.0
c = 10

d1 = a*b + c
# a += h
# b += h
c += h
d2 = a*b + c

print('d1', d1)
print('d2', d2)
print('slope', (d2 - d1)/h)


In [ ]:
class Value:

    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f'Value(data={self.data}, label={self.label})'

    def __add__(self, other):
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward

        return out

    def __mul__(self, other):
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward

        return out

    def tanh(self):
        x = self.data
        t = (math.exp(2 * x) - 1) / (math.exp(2 * x) + 1)
        out = Value(t, (self,), 'tanh')

        def _backward():
            self.grad += (1 - out.data**2) * out.grad
        out._backward = _backward

        return out

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)

        build_topo(self)

        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

a = Value(2.0, label='a')
b = Value(-3.0, label='b')
c = Value(10.0, label='c')
e = a*b; e.label = 'e'
d = e + c; d.label = 'd'
f = Value(-2.0, label='f')
L = d * f; L.label = 'L'
L

In [ ]:
from graphviz import Digraph

def trace(root):
  # builds a set of all nodes and edges in a graph
  nodes, edges = set(), set()
  def build(v):
    if v not in nodes:
      nodes.add(v)
      for child in v._prev:
        edges.add((child, v))
        build(child)
  build(root)
  return nodes, edges

def draw_dot(root):
  dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR = left to right

  nodes, edges = trace(root)
  for n in nodes:
    uid = str(id(n))
    # for any value in the graph, create a rectangular ('record') node for it
    dot.node(name = uid, label = "{ %s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad), shape='record')
    if n._op:
      # if this value is a result of some operation, create an op node for it
      dot.node(name = uid + n._op, label = n._op)
      # and connect this node to it
      dot.edge(uid + n._op, uid)

  for n1, n2 in edges:
    # connect n1 to the op node of n2
    dot.edge(str(id(n1)), str(id(n2)) + n2._op)

  return dot

In [ ]:
draw_dot(L)

In [ ]:
# Manual gradient calculation (redraw the graph above to see the changes)

# a = Value(2.0, label='a')
# b = Value(-3.0, label='b')
# c = Value(10.0, label='c')
# e = a*b; e.label = 'e'
# d = e + c; d.label = 'd'
# f = Value(-2.0, label='f')
# L = d * f; L.label = 'L'

L.grad = 1.0  # derivative of an argument itself
f.grad = d.data  # derivative of d * f with respect to f
d.grad = f.data  # derivative of d * f with respect to d

# The chain rule
c.grad = d.grad * 1.0  # (dL / dd) * (dd / dc) = d.data * 1.0: local grads multiplies
e.grad = d.grad * 1.0  # (dL / dd) * (dd / de) = d.data * 1.0: local grads multiplies
a.grad = e.grad * b.data  # (dL / de) * (de / da) = e.grad * b.data: prev global grad multiplies local grad
b.grad = e.grad * a.data  # (dL / de) * (de / db) = e.grad * a.data: prev global grad multiplies local grad

In [ ]:
# Manual influence on L (an example)

# Change inputs
a.data += 0.01 * a.grad
b.data += 0.01 * b.grad
c.data += 0.01 * c.grad
f.data += 0.01 * f.grad

# Perform forward pass
e = a * b
d = e + c
L = d * f

print(L)


In [ ]:
def lol():

    h = 0.001

    a = Value(2.0, label='a')
    b = Value(-3.0, label='b')
    c = Value(10.0, label='c')
    e = a*b; e.label = 'e'
    d = e + c; d.label = 'd'
    f = Value(-2.0, label='f')
    L = d * f; L.label = 'L'
    L1 = L.data

    a = Value(2.0, label='a')
    b = Value(-3.0, label='b')
    c = Value(10.0, label='c')
    e = a*b; e.label = 'e'
    d = e + c; d.label = 'd'
    f = Value(-2.0, label='f')
    L = d * f; L.label = 'L'
    L2 = L.data

    print((L2 - L1)/h)

lol()

In [ ]:
plt.plot(np.arange(-5, 5, 0.2), np.tanh(np.arange(-5, 5, 0.2))); plt.grid(True)

In [ ]:
# inputs x1,x2
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')
# weights w1,w2
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')
# bias of the neuron
b = Value(6.8813735870195432, label='b')
# x1*w1 + x2*w2 + b
x1w1 = x1*w1; x1w1.label = 'x1*w1'
x2w2 = x2*w2; x2w2.label = 'x2*w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1*w1 + x2*w2'
n = x1w1x2w2 + b; n.label = 'n'
o = n.tanh(); o.label = 'o'

In [ ]:
draw_dot(o)

### Calculate the gradients of the simple neuron using backward method

In [ ]:
# Calculate the gradients of the simple neuron using backward method
o.backward()

### Calculate the gradients of the simple neuron using topological order and a loop

In [ ]:
o.grad = 1.0

topo = []
visited = set()
def build_topo(v):
  if v not in visited:
    visited.add(v)
    for child in v._prev:
      build_topo(child)
    topo.append(v)

build_topo(o)

for node in reversed(topo):
  node._backward()

### Calculate the gradients of the simple neuron using _backward function one by one

In [ ]:
# o.grad = 1.0
# o._backward()

In [ ]:
# n._backward()

In [ ]:
# b._backward()
# x1w1x2w2._backward()

In [ ]:
# x1w1._backward()
# x2w2._backward()

### Calculate the gradients of the simple neuron manually

In [ ]:
# # Calculate the gradients of the simple neuron manually
# o.grad = 1.0

# n.grad = 1 - o.data**2  # do / dn = 1 - tanh(n)**2 = 1 - o.data**2

# x1w1x2w2.grad = n.grad  # '+' just distribute the same gradient back
# b.grad = n.grad # '+' just distribute the same gradient back

# x1w1.grad = x1w1x2w2.grad  # '+' just distribute the same gradient back
# x2w2.grad = x1w1x2w2.grad  # '+' just distribute the same gradient back

# x1.grad = w1.data * x1w1.grad  # (do / dx1w1) * (dx1w1 / dx1) = x1w1.grad * w1.data
# w1.grad = x1.data * x1w1.grad  # (do / dx1w1) * (dx1w1 / dw1) = x1w1.grad * x1.data
# x2.grad = w2.data * x2w2.grad  # (do / dx1w1) * (dx1w1 / dx1) = x2w2.grad * w2.data
# w2.grad = x2.data * x2w2.grad  # (do / dx1w1) * (dx1w1 / dw1) = x2w2.grad * x2.data


In [ ]:
# # Manual influence check (example)
# w1.data += 0.01 * w1.grad

# # Forward pass
# x1w1 = x1*w1
# x2w2 = x2*w2
# x1w1x2w2 = x1w1 + x2w2
# n = x1w1x2w2 + b
# o = n.tanh()

# print(o)